# AI AINAK — Notebook 2: Environmental Audio Module

**Branch 2:** Audio Ingestion, 32 kHz Preprocessing, PANNs CNN14 Bottleneck Feature Extraction, Downstream AudioMLP ESC-50 Classification, and Silence Handling.

---

## 1. Dataset Taxonomy & Asset Specification

This notebook details every dataset connected to the acoustic perception branch of AI AINAK:

| Category | Dataset Name | Source / URL | License | Purpose in AI AINAK |
| :--- | :--- | :--- | :--- | :--- |
| **Training Dataset (Backbone)** | **Google AudioSet** | [research.google.com/audioset](https://research.google.com/audioset/) | AudioSet terms | Pre-training the PANNs CNN14 acoustic backbone across 527 sound classes (2M+ 10s YouTube audio clips). *Omitted from demo due to terabyte scale; represented by `Cnn14_mAP=0.431.pth`.* |
| **Training Dataset (Downstream)** | **ESC-50 (Folds 1–4)** | [karolpiczak/ESC-50](https://github.com/karolpiczak/ESC-50) | CC BY-NC 3.0 | 1,600 5-second environmental recordings across 50 classes (sirens, car horns, footsteps, crying baby, dog barking, engine sounds). Used to train the downstream PyTorch AudioMLP. Precomputed embeddings cached in `artifacts/audio/esc50_embeddings_cnn14.npz`. |
| **Evaluation Dataset** | **ESC-50 Held-Out Fold 5** | [karolpiczak/ESC-50](https://github.com/karolpiczak/ESC-50) | CC BY-NC 3.0 | 400 held-out clips used for offline benchmark evaluation. AudioMLP achieves **89.75% accuracy and 89.59% Macro-F1** (reported in `reports/audio-downstream-eval.json`). Noise robustness tested across clean, +20 dB SNR, +10 dB SNR, and digital silence. *Optional download snippet provided below.* |
| **Demonstration Input** | **Paired Session `session-multimodal-01`** | Project Repository / Google Drive | Project License | Real-world walking corridor audio (`audio.wav`, 224 KB, 32 kHz mono PCM, 112,000 samples, 3.50s duration). Synchronized with paired video starting at $t=0.0\,	ext{s}$ per `recording.json`. |

---

## 2. Checkpoint & Demonstration Media Manifest

| Asset Name | Target Path | File Size | Origin / Acquisition Method |
| :--- | :--- | :--- | :--- |
| **PANNs CNN14 Backbone** | `checkpoints/Cnn14_mAP=0.431.pth` | ~327.43 MB | Public official checkpoint (Zenodo record 3987831). Auto-downloaded below if missing. |
| **Downstream AudioMLP** | `checkpoints/downstream_mlp.pt` | ~2.17 MB | Trained AI AINAK project checkpoint (2-layer MLP with BatchNorm and Dropout). |
| **Demonstration Audio** | `datasets/recordings/session-multimodal-01/audio.wav` | ~224 KB | Synchronized session recording audio. |
| **Session Metadata** | `datasets/recordings/session-multimodal-01/recording.json` | ~837 B | Verified session clock and sample rate assertions. |


In [ ]:
# Step 1: Install Dependencies
!pip install -q panns-inference soundfile librosa torch numpy matplotlib


In [ ]:
# Environment Detection, Google Drive Setup & Path Configuration
import os, sys, shutil, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("=" * 70)
    print("  AI AINAK: Google Colab Runtime Environment Detected")
    print("=" * 70)
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        print("  [OK] Google Drive mounted successfully at /content/drive")
    except Exception as e:
        print(f"  [NOTICE] Drive mount skipped or already mounted: {e}")

    # Standard Google Drive project folder
    DRIVE_AINAK = Path("/content/drive/MyDrive/AINAK_Colab")
    
    # Candidate project locations in priority order
    drive_candidates = [
        DRIVE_AINAK,
        Path("/content/drive/MyDrive/AINAK"),
        Path("/content/AINAK"),
        Path("/content"),
    ]
    PROJECT_ROOT = next((p for p in drive_candidates if (p / "artifacts").is_dir() or (p / "checkpoints").is_dir()), DRIVE_AINAK)
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    print("=" * 70)
    print("  AI AINAK: Local Runtime Environment Detected")
    print("=" * 70)
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"  Project Root resolved to: {PROJECT_ROOT}")

# Create standard subdirectories if not present
(PROJECT_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "datasets" / "recordings" / "session-multimodal-01").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR = PROJECT_ROOT / "outputs"



In [ ]:
# Checkpoint & Dataset Acquisition Utilities
def download_url(url: str, dest_path: Path, desc: str = "") -> Path:
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.is_file() and dest_path.stat().st_size > 0:
        print(f"  [FOUND] {desc or dest_path.name} already present ({dest_path.stat().st_size / 1e6:.2f} MB). Skipping download.")
        return dest_path
    print(f"  [DOWNLOADING] {desc or dest_path.name} from:
    {url}")
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (AINAK-Colab-Environment)"})
        with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as f:
            shutil.copyfileobj(resp, f)
        tmp_path.replace(dest_path)
        print(f"  [DOWNLOAD COMPLETE] {dest_path.name} ({dest_path.stat().st_size / 1e6:.2f} MB)")
        return dest_path
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise RuntimeError(f"Failed to download {dest_path.name} from {url}: {e}")

def find_checkpoint(filename: str, subfolder: str = "") -> Path:
    candidates = [
        PROJECT_ROOT / "artifacts" / subfolder / filename,
        PROJECT_ROOT / "artifacts" / filename,
        PROJECT_ROOT / "checkpoints" / subfolder / filename,
        PROJECT_ROOT / "checkpoints" / filename,
        PROJECT_ROOT / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[2] # Return standard checkpoints/ location as target

def find_session_file(session_id: str, filename: str) -> Path:
    candidates = [
        PROJECT_ROOT / "datasets" / "recordings" / session_id / filename,
        PROJECT_ROOT / "datasets" / session_id / filename,
        PROJECT_ROOT / session_id / filename,
        Path(filename),
    ]
    for c in candidates:
        if c.is_file() and c.stat().st_size > 0:
            return c.resolve()
    return candidates[0] # Return standard datasets/recordings/<session>/ location

def unpack_starter_bundle_if_available():
    bundle_candidates = [
        PROJECT_ROOT / "ainak_colab_starter_bundle.zip",
        PROJECT_ROOT / "outputs" / "ainak_colab_starter_bundle.zip",
        Path("/content/ainak_colab_starter_bundle.zip"),
    ]
    for b in bundle_candidates:
        if b.is_file() and b.stat().st_size > 0:
            print(f"  [UNPACKING] Found starter bundle at {b}. Extracting to {PROJECT_ROOT}...")
            import zipfile
            with zipfile.ZipFile(str(b), "r") as zf:
                zf.extractall(str(PROJECT_ROOT))
            print("  [UNPACK COMPLETE] Starter checkpoints and demonstration media extracted.")
            return True
    return False

def run_preflight_audit(required_items):
    print("=" * 80)
    print("  AI AINAK PRE-FLIGHT ASSET & DATASET INTEGRITY AUDIT")
    print("=" * 80)
    all_ok = True
    for item in required_items:
        path = Path(item["path"])
        min_size = item.get("min_size", 1)
        role = item.get("role", "Asset")
        desc = item.get("desc", path.name)
        
        if path.is_file() and path.stat().st_size >= min_size:
            size_mb = path.stat().st_size / 1e6
            status = "PASSED"
            print(f"  [OK] {status:<6} | {path.name:<28} | {size_mb:6.2f} MB | {role:<16} | {desc}")
        else:
            all_ok = False
            status = "FAILED"
            print(f"  [!!] {status:<6} | {path.name:<28} | MISSING   | {role:<16} | {desc}")
            print(f"       -> Expected location: {path}")
            if "download_url" in item:
                print(f"       -> Download URL: {item['download_url']}")
            elif "action" in item:
                print(f"       -> Action required: {item['action']}")
    print("=" * 80)
    if not all_ok:
        raise FileNotFoundError(
            "Pre-flight audit failed: One or more required datasets or model checkpoints are missing. "
            "Please review the missing items above and obtain them before proceeding."
        )
    print("  All required assets verified. Proceeding to inference...
")



## Step 2: Automated Dataset & Checkpoint Acquisition
Here we acquire the public PANNs CNN14 backbone weights from Zenodo automatically and resolve the AudioMLP checkpoint and session audio.


In [ ]:
# Check for starter bundle first
unpack_starter_bundle_if_available()

# 1. Acquire PANNs CNN14 Backbone (Official Zenodo Repository)
CNN14_URL = "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth"
cnn14_path = find_checkpoint("Cnn14_mAP=0.431.pth", "audio")
if not cnn14_path.is_file():
    cnn14_target = PROJECT_ROOT / "checkpoints" / "Cnn14_mAP=0.431.pth"
    cnn14_path = download_url(CNN14_URL, cnn14_target, "PANNs CNN14 Backbone Checkpoint")

# 2. Resolve Downstream AudioMLP & Session Audio
mlp_path = find_checkpoint("downstream_mlp.pt", "audio")
audio_path = find_session_file("session-multimodal-01", "audio.wav")
metadata_path = find_session_file("session-multimodal-01", "recording.json")

print(f"CNN14 Backbone Path: {cnn14_path}")
print(f"AudioMLP Checkpoint Path: {mlp_path}")
print(f"Session Audio Path: {audio_path}")


### Optional: Evaluation Dataset Acquisition (ESC-50)
If you wish to re-evaluate the AudioMLP classifier on the complete 2,000-clip ESC-50 dataset (or held-out fold 5), set `DOWNLOAD_ESC50_EVAL = True`. Otherwise, demonstration mode uses the synchronized session audio only.


In [ ]:
DOWNLOAD_ESC50_EVAL = False

if DOWNLOAD_ESC50_EVAL:
    esc50_zip = PROJECT_ROOT / "datasets" / "esc50.zip"
    esc50_dir = PROJECT_ROOT / "datasets" / "ESC-50"
    ESC50_URL = "https://github.com/karolpiczak/ESC-50/archive/master.zip"
    download_url(ESC50_URL, esc50_zip, "ESC-50 Environmental Audio Dataset")
    if not esc50_dir.is_dir():
        import zipfile
        with zipfile.ZipFile(str(esc50_zip), "r") as zf:
            zf.extractall(str(PROJECT_ROOT / "datasets"))
        print(f"[OK] Extracted ESC-50 dataset to: {esc50_dir}")
else:
    print("[INFO] Skipping full ESC-50 dataset download (demonstration mode active).")


## Step 3: Pre-Flight Asset Integrity & Clock Validation
We verify that all required audio models and media files exist and validate waveform sample rates.


In [ ]:
import soundfile as sf

# Execute pre-flight integrity audit
required_manifest = [
    {
        "path": cnn14_path,
        "min_size": 300_000_000,
        "role": "Backbone Weights",
        "desc": "PANNs CNN14 pretrained acoustic backbone",
        "download_url": CNN14_URL
    },
    {
        "path": mlp_path,
        "min_size": 1_000_000,
        "role": "Classifier Checkpoint",
        "desc": "Trained PyTorch AudioMLP (50 classes)",
        "action": "Upload checkpoints/downstream_mlp.pt to Google Drive or project folder"
    },
    {
        "path": audio_path,
        "min_size": 50_000,
        "role": "Demonstration Input",
        "desc": "Paired session audio (session-multimodal-01)",
        "action": "Upload session-multimodal-01/audio.wav to Google Drive or project folder"
    },
    {
        "path": metadata_path,
        "min_size": 100,
        "role": "Clock Metadata",
        "desc": "Session clock & pairing assertion",
        "action": "Ensure recording.json is present in session-multimodal-01"
    }
]
run_preflight_audit(required_manifest)

# Validate Audio Waveform Properties
info = sf.info(str(audio_path))
print(f"Audio Properties: Sample Rate: {info.samplerate} Hz | Channels: {info.channels} | Duration: {info.duration:.2f}s | Frames: {info.frames}")
assert info.duration > 1.0, f"Audio duration too short: {info.duration}s"



## Step 4: Downstream AudioMLP Architecture Definition
Matches `ainak.models.audio.downstream_classifier.AudioMLP`.


In [ ]:
import torch
from torch import nn

class AudioMLP(nn.Module):
    def __init__(self, input_dim: int = 2048, hidden_dim: int = 256, num_classes: int = 50, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Load trained AudioMLP weights
mlp_checkpoint = torch.load(str(mlp_path), map_location="cpu", weights_only=False)
esc50_classes = mlp_checkpoint["classes"]
audio_mlp = AudioMLP(input_dim=2048, hidden_dim=256, num_classes=len(esc50_classes))
audio_mlp.load_state_dict(mlp_checkpoint["state_dict"])
audio_mlp.eval()

print(f"AudioMLP initialized successfully with {len(esc50_classes)} classes.")
print(f"Sample classes: {esc50_classes[:8]}")


## Step 5: Audio Loading, Preprocessing & Sliding Windows
Loads the recording at 32 kHz and computes 2.0s windows with 1.0s stride.


In [ ]:
import librosa
import numpy as np

# Load audio waveform
waveform, sr = sf.read(str(audio_path), dtype="float32")
if waveform.ndim > 1:
    waveform = waveform.mean(axis=1) # Mono conversion

# Explicit 32 kHz resample
if sr != 32000:
    waveform = librosa.resample(waveform, orig_sr=sr, target_sr=32000)
    sr = 32000

duration = len(waveform) / sr
print(f"Audio loaded: {len(waveform)} samples @ {sr} Hz ({duration:.2f} seconds)")

# Sliding window parameters
window_sec = 2.0
hop_sec = 1.0
window_samples = int(window_sec * sr)
hop_samples = int(hop_sec * sr)

windows = []
window_timestamps = []

for start in range(0, max(1, len(waveform) - window_samples + 1), hop_samples):
    chunk = waveform[start : start + window_samples]
    if len(chunk) < window_samples:
        chunk = np.pad(chunk, (0, window_samples - len(chunk)))
    windows.append(chunk)
    window_timestamps.append(start / sr)

print(f"Extracted {len(windows)} temporal audio windows.")


## Step 6: PANNs CNN14 Inference & Downstream Classification
Extracts 2,048-D bottleneck representation and runs AudioMLP classification.


In [ ]:
from panns_inference import AudioTagging
import time

# Initialize PANNs
tagger = AudioTagging(checkpoint_path=str(cnn14_path), device="cpu")

timestamps = []
embeddings = []
predicted_classes = []
confidences = []
is_silent_list = []
latencies_ms = []

print("Running CNN14 + AudioMLP inference on audio windows...")
for idx, (ts, chunk) in enumerate(zip(window_timestamps, windows)):
    t_start = time.perf_counter()
    
    # RMS silence check
    rms = float(np.sqrt(np.mean(chunk * chunk)))
    if rms < 1e-5:
        emb_2048 = np.zeros(2048, dtype=np.float32)
        top_cls = "silence"
        top_conf = 0.0
        is_sil = True
    else:
        clipwise, emb = tagger.inference(chunk[None, :])
        emb_2048 = np.asarray(emb).reshape(-1).astype(np.float32)
        
        with torch.no_grad():
            x_tensor = torch.from_numpy(emb_2048[None, :])
            logits = audio_mlp(x_tensor)
            probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        
        top_idx = int(probs.argmax())
        top_cls = esc50_classes[top_idx]
        top_conf = float(probs[top_idx])
        is_sil = False

    t_latency = (time.perf_counter() - t_start) * 1000

    timestamps.append(ts)
    embeddings.append(emb_2048)
    predicted_classes.append(top_cls)
    confidences.append(top_conf)
    is_silent_list.append(is_sil)
    latencies_ms.append(t_latency)

    print(f"Window {idx+1} @ {ts:.2f}s: Class='{top_cls}' (Conf={top_conf*100:.1f}%) | Latency={t_latency:.1f}ms")

print(f"\nMean Audio Latency: {np.mean(latencies_ms):.2f} ms")


## Step 7: Waveform & Spectrogram Visualization


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

# Waveform
t_axis = np.linspace(0, duration, len(waveform))
ax1.plot(t_axis, waveform, color="#f59e0b", lw=0.8)
ax1.set_title("Environmental Audio Waveform (32 kHz)", fontsize=11, fontweight="bold")
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Amplitude")
ax1.grid(True, alpha=0.3)

# Spectrogram
S = librosa.feature.melspectrogram(y=waveform, sr=sr, n_mels=64)
S_dB = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_dB, x_axis="time", y_axis="mel", sr=sr, ax=ax2, cmap="magma")
fig.colorbar(img, ax=ax2, format="%+2.0f dB")
ax2.set_title("Log-Mel Spectrogram", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


## Step 8: Export Audio Outputs to `audio_outputs.npz`


In [ ]:
audio_out_path = OUTPUTS_DIR / "audio_outputs.npz"

np.savez_compressed(
    str(audio_out_path),
    timestamps=np.array(timestamps, dtype=np.float64),
    features=np.array(embeddings, dtype=np.float32),
    predicted_classes=np.array(predicted_classes),
    confidences=np.array(confidences, dtype=np.float32),
    is_silent=np.array(is_silent_list, dtype=bool),
    latencies_ms=np.array(latencies_ms, dtype=np.float32)
)

print(f"Successfully exported audio outputs to: {audio_out_path}")
print(f"File size: {audio_out_path.stat().st_size / 1e3:.1f} KB")
print(f"Feature matrix shape: {np.array(embeddings).shape} (Matches 2,048-D Transformer contract)")
